# End-to-End Audit: Build Plan vs. Codebase

This notebook audits the current implementation of the AkshHealth platform against the 12-month execution roadmap Excel build plan. It identifies what is built, what is missing, and generates a structured audit report for engineering and quality analysis.

## 1. Import Required Libraries
Import libraries for Excel processing, file operations, and code analysis.

In [ ]:
import os
import re
import ast
import pandas as pd
from openpyxl import load_workbook
from collections import defaultdict

## 2. Load and Inspect the Excel Build Plan
Load the 12-month execution roadmap and display available sheets and columns.

In [ ]:
excel_path = 'AkshHealth_12Month_ExecutionRoadmap.xlsx'
wb = load_workbook(excel_path, read_only=True)
sheet_names = wb.sheetnames
print('Available sheets:', sheet_names)

# Display first few rows of the main sheet (assume first sheet is main)
df_plan = pd.read_excel(excel_path, sheet_name=sheet_names[0])
df_plan.head()

## 3. Extract Feature List from Build Plan
Parse the build plan to extract a structured list of features, modules, or requirements.

In [ ]:
# Try to extract columns that look like features/modules/requirements
feature_columns = [col for col in df_plan.columns if any(x in col.lower() for x in ['feature', 'module', 'requirement', 'description', 'task'])]
features = df_plan[feature_columns].dropna(how='all')
features.head()

## 4. Analyze Existing Source Code
Scan the source code files to extract implemented features, classes, and functions.

In [ ]:
# Recursively scan for .ts, .tsx files in app/ and components/
def extract_code_features(root_dirs=['app', 'components']):
    code_features = defaultdict(list)
    for root_dir in root_dirs:
        for dirpath, _, filenames in os.walk(root_dir):
            for fname in filenames:
                if fname.endswith(('.ts', '.tsx')):
                    fpath = os.path.join(dirpath, fname)
                    with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()
                        # Extract function, class, and component names
                        for match in re.findall(r'(?:function|class|const|let|var)\s+([A-Za-z0-9_]+)', content):
                            code_features[fpath].append(match)
                        # Extract comments/docstrings
                        for comment in re.findall(r'\/\/.*|\/\*[\s\S]*?\*\/', content):
                            code_features[fpath].append(comment.strip())
    return code_features

code_features = extract_code_features()
list(code_features.items())[:3]  # Show sample

## 5. Map Implemented Features to Build Plan
Attempt to match features from the codebase to those in the build plan by name or description.

In [ ]:
# Attempt to match features by name/description
implemented = []
missing = []
for idx, row in features.iterrows():
    found = False
    for codefile, items in code_features.items():
        for item in items:
            if any(str(row[col]).lower() in str(item).lower() for col in feature_columns if pd.notnull(row[col])):
                implemented.append((row.to_dict(), codefile, item))
                found = True
                break
        if found:
            break
    if not found:
        missing.append(row.to_dict())

print(f"Implemented: {len(implemented)} | Missing: {len(missing)}")

## 6. Identify Missing Features
Highlight features from the build plan that are not found in the codebase.

In [ ]:
import json
print('Missing features:')
print(json.dumps(missing, indent=2))

## 7. Generate Audit Report
Summarize implemented and missing features, and highlight discrepancies.

In [ ]:
print('--- AUDIT REPORT ---')
print(f"Total features in build plan: {len(features)}")
print(f"Implemented features: {len(implemented)}")
print(f"Missing features: {len(missing)}")

print('\nSample Implemented Features:')
for feat, codefile, item in implemented[:5]:
    print(f"- {feat} in {codefile} (matched: {item})")

print('\nSample Missing Features:')
for feat in missing[:5]:
    print(f"- {feat}")